# Model Prediction Pipeline

This notebook handles:
- Loading trained models
- Making predictions on new data
- Prediction visualization
- Export results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

## 1. Load Model and Data

In [ ]:
with open('models/best_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('models/best_model_name.txt', 'r') as f:
    model_name = f.read().strip()

print(f'✓ Loaded model: {model_name}')

In [ ]:
X_test = pd.read_csv('data/X_test.csv')
y_test = pd.read_csv('data/y_test.csv').squeeze()

print(f'Test set shape: {X_test.shape}')
print(f'Target shape: {y_test.shape}')

## 2. Make Predictions

In [ ]:
y_pred = model.predict(X_test)
print(f'✓ Generated {len(y_pred)} predictions')
print(f'\nPrediction statistics:')
print(f'  Min: {y_pred.min():.4f}')
print(f'  Max: {y_pred.max():.4f}')
print(f'  Mean: {y_pred.mean():.4f}')
print(f'  Std: {y_pred.std():.4f}')

## 3. Evaluation Metrics

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f'\n=== PREDICTION METRICS ===')
print(f'Model: {model_name}')
print(f'\nR² Score: {r2:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE: {mae:.4f}')
print(f'MAPE: {mape:.2f}%')

## 4. Prediction vs Actual Comparison

In [ ]:
results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Error': y_test.values - y_pred,
    'Absolute_Error': np.abs(y_test.values - y_pred),
    'Percentage_Error': np.abs((y_test.values - y_pred) / y_test.values) * 100
})

print('\nPrediction Results Sample:')
print(results_df.head(10).to_string())
print(f'\n... (Total {len(results_df)} predictions) ...')

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(y_test, y_pred, alpha=0.5, color='#ed53a3')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Streams', fontweight='bold')
axes[0, 0].set_ylabel('Predicted Streams', fontweight='bold')
axes[0, 0].set_title('Actual vs Predicted', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(results_df['Error'], bins=50, color='#ed53a3', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Prediction Error', fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontweight='bold')
axes[0, 1].set_title('Error Distribution', fontweight='bold')
axes[0, 1].axvline(x=0, color='r', linestyle='--', linewidth=2)

axes[1, 0].scatter(y_test, results_df['Absolute_Error'], alpha=0.5, color='#ed53a3')
axes[1, 0].set_xlabel('Actual Streams', fontweight='bold')
axes[1, 0].set_ylabel('Absolute Error', fontweight='bold')
axes[1, 0].set_title('Error vs Actual Values', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(results_df['Percentage_Error'], bins=50, color='#ed53a3', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Percentage Error (%)', fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontweight='bold')
axes[1, 1].set_title('Percentage Error Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('results/predictions_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved predictions analysis plot')

## 6. Residuals Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, results_df['Error'], alpha=0.5, color='#ed53a3')
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Values', fontweight='bold')
axes[0].set_ylabel('Residuals', fontweight='bold')
axes[0].set_title('Residual Plot', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].hist(results_df['Error'], bins=50, color='#ed53a3', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Residuals', fontweight='bold')
axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)

plt.tight_layout()
plt.savefig('results/residuals_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Saved residuals analysis plot')

## 7. Export Results

In [ ]:
results_df.to_csv('results/predictions_detailed.csv', index=False)
print('✓ Saved detailed predictions to CSV')

In [ ]:
summary = pd.DataFrame({
    'Metric': ['R² Score', 'RMSE', 'MAE', 'MAPE', 'Mean Error', 'Median Error'],
    'Value': [
        r2,
        rmse,
        mae,
        mape,
        results_df['Error'].mean(),
        results_df['Error'].median()
    ]
})

summary.to_csv('results/prediction_summary.csv', index=False)
print('✓ Saved prediction summary')
print('\n=== SUMMARY ===')
print(summary.to_string(index=False))

## 8. Prediction on New Data Template

In [ ]:
def predict_new_track(features_dict, scaler):
    """
    Make predictions on a new track.
    
    Parameters:
    -----------
    features_dict : dict
        Dictionary with feature values
    scaler : StandardScaler
        Pre-fitted scaler
    
    Returns:
    --------
    float : Predicted streams (in millions)
    """
    df = pd.DataFrame([features_dict])
    df_scaled = scaler.transform(df)
    prediction = model.predict(df_scaled)[0]
    return prediction

with open('data/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

example_features = {
    'Danceability': 75,
    'Energy': 80,
    'Key': 5,
    'Loudness': -5.0,
    'Speechiness': 0.05,
    'Acousticness': 0.1,
    'Instrumentalness': 0.01,
    'Liveness': 0.15,
    'Valence': 0.65,
    'Tempo': 120,
    'Duration_ms': 210000,
    'Views': 100,
    'Likes': 2,
    'Comments': 0.5,
    'Engagement_Rate': 2.0,
    'Discussion_Rate': 0.5,
    'Like_Comment_Ratio': 4.0
}

predicted_streams = predict_new_track(example_features, scaler)
print(f'\nExample Prediction:')
print(f'Predicted Streams: {predicted_streams:.2f} million')

## 9. Confidence Intervals (Optional)

In [ ]:
residuals = results_df['Error'].values
std_error = np.std(residuals)
confidence_95 = 1.96 * std_error

results_df['Lower_Bound_95'] = results_df['Predicted'] - confidence_95
results_df['Upper_Bound_95'] = results_df['Predicted'] + confidence_95

within_ci = ((results_df['Actual'] >= results_df['Lower_Bound_95']) & 
              (results_df['Actual'] <= results_df['Upper_Bound_95'])).sum()

print(f'\n95% Confidence Interval Analysis:')
print(f'Std Error: {std_error:.4f}')
print(f'CI Width: ±{confidence_95:.4f}')
print(f'Actual values within 95% CI: {within_ci}/{len(results_df)} ({within_ci/len(results_df)*100:.1f}%)')